In [1]:
from pathlib import Path
import pandas as pd

# 1. Khai báo đường dẫn dự án
PROJECT_DIR = Path.cwd().parent

SQL_DIR = PROJECT_DIR / "results" / "sql"
KG_DIR = PROJECT_DIR / "results" / "kg"
LLM_DIR = PROJECT_DIR / "results" / "llm"

EVAL_DIR = PROJECT_DIR / "results" / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# 2. Đọc kết quả sinh truy vấn của Qwen
sql_generated = pd.read_csv(
    LLM_DIR / "text_to_sql_results.csv"
)

sparql_generated = pd.read_csv(
    LLM_DIR / "text_to_sparql_results.csv"
)

# 3. Đọc log thực thi truy vấn tham chiếu
sql_reference_log = pd.read_csv(
    SQL_DIR / "sql_execution_log.csv"
)

sparql_reference_log = pd.read_csv(
    KG_DIR / "sparql_execution_log.csv"
)

# 4. Kiểm tra hai nhánh sử dụng cùng bộ câu hỏi
assert len(sql_generated) == 44
assert len(sparql_generated) == 44

assert set(sql_generated["ID"]) == set(sparql_generated["ID"])

# 5. Hiển thị kết quả
print("=== EVALUATION DATA ===")
print("Generated SQL:", len(sql_generated))
print("Generated SPARQL:", len(sparql_generated))
print("SQL References:", len(sql_reference_log))
print("SPARQL References:", len(sparql_reference_log))

print("\n=== SQL GENERATION STATUS ===")
print(sql_generated["Status"].value_counts())

print("\n=== SPARQL GENERATION STATUS ===")
print(sparql_generated["Status"].value_counts())

print("\nEvaluation directory:", EVAL_DIR)

=== EVALUATION DATA ===
Generated SQL: 44
Generated SPARQL: 44
SQL References: 45
SPARQL References: 44

=== SQL GENERATION STATUS ===
Status
OK    44
Name: count, dtype: int64

=== SPARQL GENERATION STATUS ===
Status
OK    44
Name: count, dtype: int64

Evaluation directory: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\evaluation


In [2]:
from rdflib import Graph, Namespace, RDF

# 1. Đọc benchmark gốc của tác giả
BENCHMARK_FILE = PROJECT_DIR / "data" / "ACME_Insurance" / "investigation" / "acme-benchmark.ttl"

benchmark = Graph()
benchmark.parse(BENCHMARK_FILE, format="turtle")

QA = Namespace("http://models.data.world/benchmarks/QandA#")
DWT = Namespace("https://templates.data.world/")

# 2. Ghép từng câu hỏi với các Reference
reference_rows = []

for inquiry in benchmark.subjects(RDF.type, QA.Inquiry):
    question = str(benchmark.value(inquiry, QA.prompt))

    sql_refs = []
    sparql_refs = []

    for ref in benchmark.objects(inquiry, QA.expects):
        if (ref, RDF.type, DWT.SqlQuery) in benchmark:
            sql_refs.append(str(ref))

        elif (ref, RDF.type, DWT.SparqlQuery) in benchmark:
            sparql_refs.append(str(ref))

    reference_rows.append({
        "Question": question,
        "SQL References": sorted(sql_refs),
        "SPARQL Reference": sparql_refs[0] if len(sparql_refs) == 1 else None
    })

reference_df = pd.DataFrame(reference_rows)

# 3. Ghép với truy vấn do Qwen sinh
evaluation_df = sql_generated[
    ["ID", "Question", "Generated SQL"]
].merge(
    sparql_generated[["ID", "Question", "Generated SPARQL"]],
    on=["ID", "Question"],
    validate="one_to_one"
)

evaluation_df = evaluation_df.merge(
    reference_df,
    on="Question",
    validate="one_to_one"
)

evaluation_df = evaluation_df.sort_values("ID").reset_index(drop=True)

# 4. Kiểm tra dữ liệu
assert len(evaluation_df) == 44
assert evaluation_df["SPARQL Reference"].notna().all()

print("=== REFERENCE MAPPING ===")
print("Total questions:", len(evaluation_df))
print("SQL References:", evaluation_df["SQL References"].apply(len).sum())
print("SPARQL References:", evaluation_df["SPARQL Reference"].notna().sum())

print("\nQuestions with multiple SQL References:")
display(evaluation_df[
    evaluation_df["SQL References"].apply(len) > 1
][["ID", "Question", "SQL References"]])

print("\nMapping preview:")
display(evaluation_df.head())

=== REFERENCE MAPPING ===
Total questions: 44
SQL References: 45
SPARQL References: 44

Questions with multiple SQL References:


,ID,Question,SQL References
31,32,Return agents and the policies they have sold ...,[https://templates.data.world/query-a810b049-4...



Mapping preview:


,ID,Question,Generated SQL,Generated SPARQL,SQL References,SPARQL Reference
0,1,What is the loss ratio of each policy and agen...,"SELECT \n p.Policy_Number,\n ap.Party_ID...",PREFIX in: <http://data.world/schema/insurance...,[https://templates.data.world/query-63dd7451-5...,https://templates.data.world/query-a916ea29-b4...
1,2,Return all the policies we have by policy numb...,"SELECT Policy_Identifier, Effective_Date, Expi...",PREFIX in: <http://data.world/schema/insurance...,[https://templates.data.world/query-5e26c413-d...,https://templates.data.world/query-481d808c-a3...
2,3,"What are the loss payment, loss reserve, expen...","SELECT \n c.Claim_Number,\n p.Policy_Num...",PREFIX in: <http://data.world/schema/insurance...,[https://templates.data.world/query-47603eb6-8...,https://templates.data.world/query-a7e183f5-33...
3,4,"What are the total loss, which is the sum of l...","SELECT \n c.Catastrophe_Name, \n p.Polic...",PREFIX in: <http://data.world/schema/insurance...,[https://templates.data.world/query-37e3d4bb-7...,https://templates.data.world/query-d1fc6863-b4...
4,5,Return agents and the policy they have sold th...,"SELECT \n a.Agent_Id,\n p.Policy_Number,...",PREFIX in: <http://data.world/schema/insurance...,[https://templates.data.world/query-425d1228-4...,https://templates.data.world/query-61bca3dc-7a...


In [3]:
import duckdb
import pandas as pd
import re

# 1. Kết nối DuckDB
DB_FILE = SQL_DIR / "acme.duckdb"
conn = duckdb.connect(str(DB_FILE), read_only=True)

# 2. Chuẩn bị nơi lưu kết quả
sql_generated_outputs = {}
sql_execution_log = []

# 3. Thực thi từng câu SQL do Qwen sinh
for _, row in evaluation_df.iterrows():
    question_id = int(row["ID"])
    sql = row["Generated SQL"]

    try:
        # Chỉ cho phép truy vấn đọc dữ liệu
        if not isinstance(sql, str) or not re.match(
            r"^\s*(SELECT|WITH)\b", sql, re.IGNORECASE
        ):
            raise ValueError("Not a SELECT query")

        # Thực thi SQL và lấy kết quả
        result = conn.execute(sql).df()

        sql_generated_outputs[question_id] = result

        status = "OK"
        rows = len(result)
        error = ""

    except Exception as e:
        status = "ERROR"
        rows = None
        error = str(e).splitlines()[0]

    # 4. Ghi nhận trạng thái thực thi
    sql_execution_log.append({
        "ID": question_id,
        "Status": status,
        "Rows": rows,
        "Error": error
    })

# 5. Đóng kết nối và tổng hợp kết quả
conn.close()

sql_execution_df = pd.DataFrame(sql_execution_log)

print("=== SQL EXECUTION RESULTS ===")
print("Total queries:", len(sql_execution_df))

print("\nExecution Status:")
print(sql_execution_df["Status"].value_counts())

print("\nEmpty Results:")
print((sql_execution_df["Rows"] == 0).sum())

print("\nFailed Queries:")
display(sql_execution_df[sql_execution_df["Status"] == "ERROR"])

=== SQL EXECUTION RESULTS ===
Total queries: 44

Execution Status:
Status
ERROR    33
OK       11
Name: count, dtype: int64

Empty Results:
3

Failed Queries:


,ID,Status,Rows,Error
0,1,ERROR,NaN,"Binder Error: Table ""apr"" does not have a colu..."
2,3,ERROR,NaN,"Binder Error: Table ""c"" does not have a column..."
3,4,ERROR,NaN,"Binder Error: Table ""lp"" does not have a colum..."
4,5,ERROR,NaN,Catalog Error: Table with name Agent does not ...
6,7,ERROR,NaN,"Binder Error: Table ""ca"" does not have a colum..."
7,8,ERROR,NaN,"Binder Error: Referenced column ""Policy_Number..."
8,9,ERROR,NaN,"Binder Error: Table ""c"" does not have a column..."
9,10,ERROR,NaN,"Binder Error: Table ""a"" does not have a column..."
10,11,ERROR,NaN,"Binder Error: Table ""Claim"" does not have a co..."
11,12,ERROR,NaN,"Binder Error: Referenced column ""Policy_Number..."


In [4]:
from rdflib import Graph
import pandas as pd

# 1. Đọc Knowledge Graph đã chuẩn hóa
KG_FILE = KG_DIR / "acme_knowledge_graph_local.nt"

kg_eval = Graph()
kg_eval.parse(str(KG_FILE), format="nt")

print("Knowledge Graph Triples:", len(kg_eval))

# 2. Chuẩn bị nơi lưu kết quả
sparql_generated_outputs = {}
sparql_execution_log = []

# 3. Thực thi từng SPARQL do Qwen sinh
for _, row in evaluation_df.iterrows():
    question_id = int(row["ID"])
    sparql = row["Generated SPARQL"]

    try:
        # Thực thi truy vấn trên Knowledge Graph
        result = kg_eval.query(sparql)

        if result.type != "SELECT":
            raise ValueError("Not a SELECT query")

        # Chuyển kết quả thành DataFrame
        columns = [str(v) for v in result.vars]

        df = pd.DataFrame([
            [
                str(value) if value is not None else None
                for value in record
            ]
            for record in result
        ], columns=columns)

        sparql_generated_outputs[question_id] = df

        status = "OK"
        rows = len(df)
        error = ""

    except Exception as e:
        status = "ERROR"
        rows = None
        error = str(e).splitlines()[0] if str(e) else type(e).__name__

    # 4. Ghi nhận kết quả thực thi
    sparql_execution_log.append({
        "ID": question_id,
        "Status": status,
        "Rows": rows,
        "Error": error
    })

# 5. Tổng hợp kết quả
sparql_execution_df = pd.DataFrame(sparql_execution_log)

print("\n=== SPARQL EXECUTION RESULTS ===")
print("Total queries:", len(sparql_execution_df))

print("\nExecution Status:")
print(sparql_execution_df["Status"].value_counts())

print("\nEmpty Results:")
print((sparql_execution_df["Rows"] == 0).sum())

print("\nFailed Queries:")
display(sparql_execution_df[
    sparql_execution_df["Status"] == "ERROR"
])

print("\nEmpty Queries:")
display(sparql_execution_df[
    sparql_execution_df["Rows"] == 0
])

Knowledge Graph Triples: 228

=== SPARQL EXECUTION RESULTS ===
Total queries: 44

Execution Status:
Status
OK       33
ERROR    11
Name: count, dtype: int64

Empty Results:
15

Failed Queries:


,ID,Status,Rows,Error
0,1,ERROR,NaN,"Expected SelectQuery, found '(' (at char 159)..."
3,4,ERROR,NaN,"Expected SelectQuery, found 'SUM' (at char 57..."
5,6,ERROR,NaN,"Expected SelectQuery, found ')' (at char 96),..."
7,8,ERROR,NaN,"Expected SelectQuery, found 'SUM' (at char 57..."
9,10,ERROR,NaN,Unknown namespace prefix : in
11,12,ERROR,NaN,"Expected SelectQuery, found 'SUM' (at char 57..."
12,13,ERROR,NaN,"Expected SelectQuery, found 'SUM' (at char 57..."
17,18,ERROR,NaN,"Expected SelectQuery, found 'AVG' (at char 57..."
26,27,ERROR,NaN,"Expected SelectQuery, found 'SUM' (at char 57..."
29,30,ERROR,NaN,'NotBoundError' object has no attribute 'datat...



Empty Queries:


,ID,Status,Rows,Error
2,3,OK,0.0,
4,5,OK,0.0,
6,7,OK,0.0,
8,9,OK,0.0,
15,16,OK,0.0,
16,17,OK,0.0,
18,19,OK,0.0,
20,21,OK,0.0,
23,24,OK,0.0,
25,26,OK,0.0,


In [5]:
import pandas as pd

# 1. Đường dẫn kết quả tham chiếu
SQL_REF_DIR = SQL_DIR / "reference_results"
SPARQL_REF_DIR = KG_DIR / "sparql_results"

# 2. Hàm đọc kết quả truy vấn
def load_result(folder, ref_id):
    filename = ref_id.split("/")[-1] + ".csv"
    return pd.read_csv(
        folder / filename,
        dtype=str,
        keep_default_na=False
    )

# 3. Hàm so sánh giá trị, bỏ qua thứ tự các dòng
def same_values(df1, df2):
    if df1.shape != df2.shape:
        return False

    rows1 = sorted(df1.apply(tuple, axis=1))
    rows2 = sorted(df2.apply(tuple, axis=1))

    return rows1 == rows2

# 4. Kiểm tra đáp án tham chiếu của từng câu hỏi
audit_rows = []

for _, row in evaluation_df.iterrows():

    sparql_result = load_result(
        SPARQL_REF_DIR, row["SPARQL Reference"]
    )

    sql_results = [
        load_result(SQL_REF_DIR, ref_id)
        for ref_id in row["SQL References"]
    ]

    # So sánh với từng SQL Reference của câu hỏi
    row_match = any(
        len(df) == len(sparql_result)
        for df in sql_results
    )

    column_match = any(
        len(df.columns) == len(sparql_result.columns)
        for df in sql_results
    )

    value_match = any(
        same_values(df, sparql_result)
        for df in sql_results
    )

    audit_rows.append({
        "ID": row["ID"],
        "SQL Rows": [len(df) for df in sql_results],
        "SPARQL Rows": len(sparql_result),
        "Row Match": row_match,
        "Column Match": column_match,
        "Exact Value Match": value_match
    })

# 5. Tổng hợp kết quả
reference_audit_df = pd.DataFrame(audit_rows)

print("=== REFERENCE AUDIT ===")
print("Total questions:", len(reference_audit_df))
print("Row count match:", reference_audit_df["Row Match"].sum())
print("Column count match:", reference_audit_df["Column Match"].sum())
print("Exact value match:", reference_audit_df["Exact Value Match"].sum())

print("\nReferences requiring review:")
display(reference_audit_df[
    ~reference_audit_df["Exact Value Match"]
])

# 6. Lưu kết quả kiểm tra
reference_audit_df.to_csv(
    EVAL_DIR / "reference_audit.csv",
    index=False
)

=== REFERENCE AUDIT ===
Total questions: 44
Row count match: 44
Column count match: 44
Exact value match: 39

References requiring review:


,ID,SQL Rows,SPARQL Rows,Row Match,Column Match,Exact Value Match
1,2,[2],2,True,True,False
5,6,[1],1,True,True,False
17,18,[1],1,True,True,False
22,23,[6],6,True,True,False
28,29,[2],2,True,True,False


In [8]:
import re
import pandas as pd
from decimal import Decimal
from datetime import datetime

# 1. Chuẩn hóa giá trị của từng ô
def normalize_value(value):
    value = str(value).strip()

    if value == "" or value.lower() in ("null", "none", "nan"):
        return None

    # Chuẩn hóa ngày và thời gian
    if re.match(r"^\d{4}-\d{2}-\d{2}", value):
        try:
            return datetime.fromisoformat(
                value.replace("Z", "+00:00")
            ).isoformat()
        except ValueError:
            pass

    # Chuẩn hóa giá trị số
    try:
        # Giữ nguyên mã định danh có số 0 ở đầu
        if re.fullmatch(r"0\d+", value):
            return value

        return Decimal(value)
    except Exception:
        return value


# 2. So sánh hai giá trị đã chuẩn hóa
def same_cell(a, b):
    a = normalize_value(a)
    b = normalize_value(b)

    if isinstance(a, Decimal) and isinstance(b, Decimal):
        return abs(a - b) <= Decimal("0.000001")

    return a == b


# 3. So sánh hai bảng, bỏ qua thứ tự dòng
def same_result(df1, df2):
    if df1.shape != df2.shape:
        return False

    remaining = df2.values.tolist()

    for row1 in df1.values.tolist():
        match = None

        for i, row2 in enumerate(remaining):
            if all(same_cell(a, b) for a, b in zip(row1, row2)):
                match = i
                break

        if match is None:
            return False

        remaining.pop(match)

    return True


# 4. So sánh lại toàn bộ đáp án tham chiếu
normalized_rows = []

for _, row in evaluation_df.iterrows():
    sparql_df = load_result(
        SPARQL_REF_DIR, row["SPARQL Reference"]
    )

    sql_dfs = [
        load_result(SQL_REF_DIR, ref_id)
        for ref_id in row["SQL References"]
    ]

    matched = any(
        same_result(sql_df, sparql_df)
        for sql_df in sql_dfs
    )

    normalized_rows.append({
        "ID": int(row["ID"]),
        "Normalized Match": matched
    })

normalized_audit_df = pd.DataFrame(normalized_rows)

# 5. Tổng hợp kết quả
print("=== NORMALIZED REFERENCE AUDIT ===")
print("Total questions:", len(normalized_audit_df))
print("Matching results:", normalized_audit_df["Normalized Match"].sum())
print("Non-matching results:", (~normalized_audit_df["Normalized Match"]).sum())

# 6. Hiển thị những câu vẫn chưa khớp
failed_ids = normalized_audit_df.loc[
    ~normalized_audit_df["Normalized Match"], "ID"
].tolist()

for question_id in failed_ids:
    row = evaluation_df.loc[
        evaluation_df["ID"] == question_id
    ].iloc[0]

    print(f"\n=== QUESTION {question_id} ===")
    print(row["Question"])

    print("\nSQL Reference:")
    for ref_id in row["SQL References"]:
        display(load_result(SQL_REF_DIR, ref_id))

    print("SPARQL Reference:")
    display(load_result(SPARQL_REF_DIR, row["SPARQL Reference"]))

# 7. Lưu kết quả kiểm tra
normalized_audit_df.to_csv(
    EVAL_DIR / "normalized_reference_audit.csv",
    index=False
)

from rdflib import URIRef, Literal
from rdflib.plugins.sparql.operators import register_custom_function

# 8. Bổ sung hàm date_diff của data.world cho RDFLib
def local_date_diff(start, end, unit):
    if str(unit).lower() != "day":
        raise ValueError("Only day is supported")

    days = (end.toPython() - start.toPython()).days
    return Literal(days)

register_custom_function(
    URIRef("http://data.world/function/functions#date_diff"),
    local_date_diff,
    override=True
)

# 9. Lấy SPARQL Reference của câu hỏi số 18
row = evaluation_df[evaluation_df["ID"] == 18].iloc[0]

ref_id = row["SPARQL Reference"]
query = str(benchmark.value(URIRef(ref_id), QA.queryText))

# Bỏ SERVICE để truy vấn trực tiếp Knowledge Graph local
query = re.sub(
    r"\bSERVICE\s+[\w-]*:mapped\s*(?=\{)",
    "",
    query,
    flags=re.IGNORECASE
)

# 10. Thực thi lại SPARQL Reference
result = kg_eval.query(query)

corrected_df = pd.DataFrame(
    [
        [str(value) if value is not None else None for value in record]
        for record in result
    ],
    columns=[str(v) for v in result.vars]
)

# 11. So sánh với SQL Reference
sql_ref = load_result(SQL_REF_DIR, row["SQL References"][0])
matched = same_result(sql_ref, corrected_df)

print("=== QUESTION 18 ===")
display(corrected_df)
print("SQL-SPARQL Match:", matched)

# 12. Lưu riêng đáp án SPARQL đã xử lý hàm mở rộng
if matched:
    corrected_dir = EVAL_DIR / "corrected_references"
    corrected_dir.mkdir(parents=True, exist_ok=True)

    corrected_df.to_csv(
        corrected_dir / (ref_id.split("/")[-1] + ".csv"),
        index=False
    )

    normalized_audit_df.loc[
        normalized_audit_df["ID"] == 18,
        "Normalized Match"
    ] = True

print("\nTotal matching references:",
      normalized_audit_df["Normalized Match"].sum())

=== NORMALIZED REFERENCE AUDIT ===
Total questions: 44
Matching results: 43
Non-matching results: 1

=== QUESTION 18 ===
What is the average time to settle a claim by policy number?

SQL Reference:


,Policy_Number,"avg(date_diff('day', claim_open_date, claim_close_date))"
0,31003000336,20.5


SPARQL Reference:


,PolicyNumber,AvgDaysToSettle
0,31003000336,0


=== QUESTION 18 ===


,PolicyNumber,AvgDaysToSettle
0,31003000336,20.5


SQL-SPARQL Match: True

Total matching references: 44


In [9]:
# 1. Chuẩn bị danh sách kết quả đánh giá
sql_scores = []

# 2. Duyệt toàn bộ 44 câu hỏi
for _, row in evaluation_df.iterrows():
    question_id = int(row["ID"])

    # Lấy trạng thái thực thi SQL
    status = sql_execution_df.loc[
        sql_execution_df["ID"] == question_id,
        "Status"
    ].iloc[0]

    correct = False

    # 3. So sánh kết quả nếu SQL thực thi thành công
    if status == "OK":
        generated = sql_generated_outputs[question_id]

        for ref_id in row["SQL References"]:
            reference = load_result(SQL_REF_DIR, ref_id)

            if same_result(generated, reference):
                correct = True
                break

    # 4. Ghi nhận kết quả từng câu hỏi
    sql_scores.append({
        "ID": question_id,
        "Question": row["Question"],
        "Execution Status": status,
        "Correct": correct
    })

# 5. Tính Execution Accuracy
sql_accuracy_df = pd.DataFrame(sql_scores)

total = len(sql_accuracy_df)
correct = int(sql_accuracy_df["Correct"].sum())
accuracy = correct / total

print("=== TEXT-TO-SQL EVALUATION ===")
print("Total questions:", total)
print("Executed successfully:", (sql_accuracy_df["Execution Status"] == "OK").sum())
print("Correct answers:", correct)
print("Incorrect answers:", total - correct)
print(f"Execution Accuracy: {accuracy:.2%}")

# 6. Hiển thị những truy vấn chạy được nhưng trả lời sai
print("\n=== EXECUTED BUT INCORRECT ===")

display(sql_accuracy_df[
    (sql_accuracy_df["Execution Status"] == "OK") &
    (~sql_accuracy_df["Correct"])
])

# 7. Lưu kết quả đánh giá
sql_accuracy_df.to_csv(
    EVAL_DIR / "text_to_sql_accuracy.csv",
    index=False
)

=== TEXT-TO-SQL EVALUATION ===
Total questions: 44
Executed successfully: 11
Correct answers: 4
Incorrect answers: 40
Execution Accuracy: 9.09%

=== EXECUTED BUT INCORRECT ===


,ID,Question,Execution Status,Correct
1,2,Return all the policies we have by policy numb...,OK,False
5,6,What is the average policy size which is the t...,OK,False
13,14,What are all the premiums that have been paid ...,OK,False
19,20,What are the expense reserve amount of all cla...,OK,False
26,27,What is the total amount of premiums that a po...,OK,False
35,36,What are the expense payment amount of all cla...,OK,False
43,44,What are the loss payment amount of all claims...,OK,False


In [10]:
# 1. Chuẩn bị danh sách kết quả đánh giá
sparql_scores = []

# Thư mục chứa Reference đã xử lý lỗi tương thích
CORRECTED_DIR = EVAL_DIR / "corrected_references"

# 2. Duyệt toàn bộ 44 câu hỏi
for _, row in evaluation_df.iterrows():
    question_id = int(row["ID"])

    # Lấy trạng thái thực thi SPARQL
    status = sparql_execution_df.loc[
        sparql_execution_df["ID"] == question_id,
        "Status"
    ].iloc[0]

    correct = False

    # 3. So sánh kết quả nếu SPARQL thực thi thành công
    if status == "OK":
        generated = sparql_generated_outputs[question_id]
        ref_id = row["SPARQL Reference"]

        # Ưu tiên Reference đã sửa nếu có
        filename = ref_id.split("/")[-1] + ".csv"

        if (CORRECTED_DIR / filename).exists():
            reference = load_result(CORRECTED_DIR, ref_id)
        else:
            reference = load_result(SPARQL_REF_DIR, ref_id)

        correct = same_result(generated, reference)

    # 4. Ghi nhận kết quả từng câu hỏi
    sparql_scores.append({
        "ID": question_id,
        "Question": row["Question"],
        "Execution Status": status,
        "Correct": correct
    })

# 5. Tính Execution Accuracy
sparql_accuracy_df = pd.DataFrame(sparql_scores)

total = len(sparql_accuracy_df)
correct = int(sparql_accuracy_df["Correct"].sum())
accuracy = correct / total

print("=== TEXT-TO-SPARQL EVALUATION ===")
print("Total questions:", total)
print("Executed successfully:", (sparql_accuracy_df["Execution Status"] == "OK").sum())
print("Correct answers:", correct)
print("Incorrect answers:", total - correct)
print(f"Execution Accuracy: {accuracy:.2%}")

# 6. Hiển thị những truy vấn chạy được nhưng trả lời sai
print("\n=== EXECUTED BUT INCORRECT ===")

display(sparql_accuracy_df[
    (sparql_accuracy_df["Execution Status"] == "OK") &
    (~sparql_accuracy_df["Correct"])
])

# 7. Lưu kết quả đánh giá
sparql_accuracy_df.to_csv(
    EVAL_DIR / "text_to_sparql_accuracy.csv",
    index=False
)

=== TEXT-TO-SPARQL EVALUATION ===
Total questions: 44
Executed successfully: 33
Correct answers: 8
Incorrect answers: 36
Execution Accuracy: 18.18%

=== EXECUTED BUT INCORRECT ===


,ID,Question,Execution Status,Correct
1,2,Return all the policies we have by policy numb...,OK,False
2,3,"What are the loss payment, loss reserve, expen...",OK,False
4,5,Return agents and the policy they have sold th...,OK,False
6,7,"What are the total loss, which is the sum of l...",OK,False
8,9,"What are the loss payment, loss reserve, expen...",OK,False
10,11,How many claims have been placed by policy num...,OK,False
13,14,What are all the premiums that have been paid ...,OK,False
14,15,"What are the loss payment, loss reserve, expen...",OK,False
15,16,Return agents and the policy they have sold th...,OK,False
16,17,Return all the policies and the agents that so...,OK,False


In [11]:
import pandas as pd

# 1. Chuẩn bị dữ liệu phân tích lỗi
error_rows = []

for method, scores, logs in [
    ("Text-to-SQL", sql_accuracy_df, sql_execution_df),
    ("Text-to-SPARQL", sparql_accuracy_df, sparql_execution_df)
]:

    # Ghép kết quả chấm điểm với log thực thi
    df = scores.merge(
        logs[["ID", "Rows", "Error"]],
        on="ID",
        validate="one_to_one"
    )

    # 2. Phân loại từng câu hỏi
    for _, row in df.iterrows():

        if row["Execution Status"] == "ERROR":
            category = "Lỗi thực thi"
        elif row["Correct"]:
            category = "Đúng"
        elif row["Rows"] == 0:
            category = "Kết quả rỗng"
        else:
            category = "Sai nội dung"

        error_rows.append({
            "Method": method,
            "ID": row["ID"],
            "Question": row["Question"],
            "Category": category,
            "Error": row["Error"]
        })

# 3. Tạo bảng phân tích
error_analysis_df = pd.DataFrame(error_rows)

categories = [
    "Đúng", "Lỗi thực thi",
    "Kết quả rỗng", "Sai nội dung"
]

summary = pd.crosstab(
    error_analysis_df["Method"],
    error_analysis_df["Category"]
).reindex(columns=categories, fill_value=0)

print("=== PHÂN TÍCH LỖI ===")
display(summary)

# 4. Hiển thị các lỗi thực thi để kiểm tra nguyên nhân
print("\n=== LỖI THỰC THI ===")
display(
    error_analysis_df[
        error_analysis_df["Category"] == "Lỗi thực thi"
    ][["Method", "ID", "Error"]]
)

# 5. Lưu kết quả phân tích
error_analysis_df.to_csv(
    EVAL_DIR / "error_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

=== PHÂN TÍCH LỖI ===


Category,Đúng,Lỗi thực thi,Kết quả rỗng,Sai nội dung
Method,,,,
Text-to-SPARQL,8,11,15,10
Text-to-SQL,4,33,3,4



=== LỖI THỰC THI ===


,Method,ID,Error
0,Text-to-SQL,1,"Binder Error: Table ""apr"" does not have a colu..."
2,Text-to-SQL,3,"Binder Error: Table ""c"" does not have a column..."
3,Text-to-SQL,4,"Binder Error: Table ""lp"" does not have a colum..."
4,Text-to-SQL,5,Catalog Error: Table with name Agent does not ...
6,Text-to-SQL,7,"Binder Error: Table ""ca"" does not have a colum..."
7,Text-to-SQL,8,"Binder Error: Referenced column ""Policy_Number..."
8,Text-to-SQL,9,"Binder Error: Table ""c"" does not have a column..."
9,Text-to-SQL,10,"Binder Error: Table ""a"" does not have a column..."
10,Text-to-SQL,11,"Binder Error: Table ""Claim"" does not have a co..."
11,Text-to-SQL,12,"Binder Error: Referenced column ""Policy_Number..."


In [12]:
import pandas as pd

# 1. Tổng hợp kết quả đánh giá của hai phương pháp
summary_rows = []

for method, scores, logs in [
    ("Text-to-SQL", sql_accuracy_df, sql_execution_df),
    ("Text-to-SPARQL", sparql_accuracy_df, sparql_execution_df)
]:
    total = len(scores)
    correct = int(scores["Correct"].sum())

    summary_rows.append({
        "Method": method,
        "Total Questions": total,
        "Executed": int((logs["Status"] == "OK").sum()),
        "Execution Errors": int((logs["Status"] == "ERROR").sum()),
        "Empty Results": int((logs["Rows"] == 0).sum()),
        "Correct": correct,
        "Incorrect": total - correct,
        "Accuracy (%)": round(correct / total * 100, 2)
    })

summary_df = pd.DataFrame(summary_rows)

# 2. Ghép kết quả đánh giá theo từng câu hỏi
final_df = evaluation_df[
    ["ID", "Question", "Generated SQL", "Generated SPARQL"]
].copy()

final_df = final_df.merge(
    sql_accuracy_df[["ID", "Execution Status", "Correct"]].rename(
        columns={
            "Execution Status": "SQL Status",
            "Correct": "SQL Correct"
        }
    ),
    on="ID",
    validate="one_to_one"
)

final_df = final_df.merge(
    sparql_accuracy_df[["ID", "Execution Status", "Correct"]].rename(
        columns={
            "Execution Status": "SPARQL Status",
            "Correct": "SPARQL Correct"
        }
    ),
    on="ID",
    validate="one_to_one"
)

final_df = final_df.sort_values("ID").reset_index(drop=True)

# 3. Kiểm tra tính đầy đủ của kết quả
assert len(final_df) == 44
assert len(normalized_audit_df) == 44
assert normalized_audit_df["Normalized Match"].all()

# 4. Lưu các bảng kết quả
summary_df.to_csv(
    EVAL_DIR / "final_summary.csv",
    index=False
)

final_df.to_csv(
    EVAL_DIR / "final_question_results.csv",
    index=False,
    encoding="utf-8-sig"
)

sql_execution_df.to_csv(
    EVAL_DIR / "sql_execution_log.csv",
    index=False
)

sparql_execution_df.to_csv(
    EVAL_DIR / "sparql_execution_log.csv",
    index=False
)

error_analysis_df.to_csv(
    EVAL_DIR / "error_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

normalized_audit_df.to_csv(
    EVAL_DIR / "normalized_reference_audit_corrected.csv",
    index=False
)

# 5. Hiển thị kết quả cuối cùng
print("=== TỔNG KẾT EXPERIMENT 05 ===")
display(summary_df)

print("\n=== PHÂN TÍCH LỖI ===")
display(summary)

# 6. So sánh độ chính xác
sql_acc = summary_df.loc[
    summary_df["Method"] == "Text-to-SQL", "Accuracy (%)"
].iloc[0]

sparql_acc = summary_df.loc[
    summary_df["Method"] == "Text-to-SPARQL", "Accuracy (%)"
].iloc[0]

print("\n=== SO SÁNH HAI PHƯƠNG PHÁP ===")
print(f"Text-to-SQL Accuracy: {sql_acc:.2f}%")
print(f"Text-to-SPARQL Accuracy: {sparql_acc:.2f}%")
print(f"Difference: {sparql_acc - sql_acc:+.2f} percentage points")

print("\nOutput directory:", EVAL_DIR)

=== TỔNG KẾT EXPERIMENT 05 ===


,Method,Total Questions,Executed,Execution Errors,Empty Results,Correct,Incorrect,Accuracy (%)
0,Text-to-SQL,44,11,33,3,4,40,9.09
1,Text-to-SPARQL,44,33,11,15,8,36,18.18



=== PHÂN TÍCH LỖI ===


Category,Đúng,Lỗi thực thi,Kết quả rỗng,Sai nội dung
Method,,,,
Text-to-SPARQL,8,11,15,10
Text-to-SQL,4,33,3,4



=== SO SÁNH HAI PHƯƠNG PHÁP ===
Text-to-SQL Accuracy: 9.09%
Text-to-SPARQL Accuracy: 18.18%
Difference: +9.09 percentage points

Output directory: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\evaluation
